# Huấn luyện mô hình Đường An

Notebook giải thích và trình bày từng bước của pipeline thực tế: đọc dữ liệu, làm sạch, chia tập, tiền xử lý, Logistic Regression, MLP, đánh giá và DQN. Các code cell được chia nhỏ để có thể trình bày tuần tự.

**Cách chạy:** chọn kernel Python có thư viện trong `requirements.txt`, mở notebook từ bên trong thư mục dự án và chạy từ trên xuống. Mặc định các cờ huấn luyện đều `False`: notebook chỉ đọc kết quả đã lưu, không huấn luyện hoặc ghi đè model. Không dùng dữ liệu cá nhân. Kết quả phục vụ học thuật, không chẩn đoán y khoa.


## 1. Chuẩn bị môi trường

Nạp thư viện và xác định gốc dự án theo vị trí thư mục hiện tại.


In [ ]:
from pathlib import Path
import json, random, logging
import numpy as np
import pandas as pd
import joblib, torch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN


### 1.1. Đường dẫn và chế độ chạy

`DATA` chứa dữ liệu gốc; `OUT` là thư mục riêng của notebook. Để huấn luyện lại, đặt **một** trong `RUN_SYMPTOMS` hoặc `RUN_BRFSS` thành `True`, chạy lại tất cả cell từ đầu. Muốn chạy cả hai bộ, chạy lần lượt mỗi lần một bộ. `RUN_RL` điều khiển DQN độc lập.


In [ ]:
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ml' / 'train.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Mở notebook bên trong thư mục project')
DATA = ROOT / 'data/raw'
OUT = ROOT / 'bao_cao_project/notebook/ket_qua_moi'
RUN_SYMPTOMS = False
RUN_BRFSS = False
RUN_RL = False
if RUN_SYMPTOMS and RUN_BRFSS:
    raise ValueError('Chỉ bật một bộ phân loại mỗi lần chạy notebook.')
RUN_CLASSIFIER = RUN_SYMPTOMS or RUN_BRFSS
kind = 'symptoms' if RUN_SYMPTOMS else 'brfss'
paths = ([str(DATA / 'diabetes_data_upload.csv')] if kind == 'symptoms' else
         [str(DATA / f'LLCP{year}.XPT') for year in (2021, 2022, 2023)])
output = str(OUT)
logging.basicConfig(level=logging.INFO)
log = logging.getLogger('training_notebook')
print('Project:', ROOT)
for path in [DATA/'diabetes_data_upload.csv', *[DATA/f'LLCP{year}.XPT' for year in (2021, 2022, 2023)]]:
    print(path.name, 'OK' if path.exists() else 'CHƯA CÓ')


## 2. Đặc trưng và chuẩn bị dữ liệu

UCI và BRFSS có nhãn, đặc trưng và cách chia khác nhau. Không gộp hai bộ thành một tập.


In [ ]:
SYMPTOMS = [
    'age', 'gender', 'polyuria', 'polydipsia', 'sudden_weight_loss', 'weakness',
    'polyphagia', 'genital_thrush', 'visual_blurring', 'itching', 'irritability',
    'delayed_healing', 'partial_paresis', 'muscle_stiffness', 'alopecia', 'obesity'
]
BRFSS = ['_AGEG5YR', '_BMI5', 'SEXVAR', '_SMOKER3', '_TOTINDA']
CODES = {
    '_AGEG5YR': list(range(1, 14)),
    'SEXVAR': [1, 2],
    '_SMOKER3': [1, 2, 3, 4],
    '_TOTINDA': [1, 2]
}


### 2.1. Định nghĩa mạng MLP

Hai lớp ẩn có 64 và 32 nút, dùng ReLU; dropout 0,2 giúp giảm phụ thuộc vào một số nút trong khi huấn luyện. Đầu ra là một logit, sigmoid được áp dụng khi đánh giá.


In [ ]:
class Network(nn.Module):
    def __init__(self, n: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


### 2.2. Đọc và chuẩn hóa UCI

Chuẩn hóa tên cột, nhãn, tuổi và giá trị dạng chuỗi. Các hồ sơ trùng và xung đột được xử lý ở bước chia tập phía dưới.


In [ ]:
def symptom_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    missing = set(SYMPTOMS + ['class']) - set(df.columns)
    if missing:
        raise ValueError(f'UCI columns missing: {sorted(missing)}')
    df = df[SYMPTOMS + ['class']].copy()
    df['class'] = df['class'].astype(str).str.strip().str.lower().map({'positive': 1, 'negative': 0})
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    for c in SYMPTOMS[1:]:
        df[c] = df[c].astype('string').str.strip().str.lower()
    return df.dropna(subset=['class'])


### 2.3. Đọc và xử lý BRFSS

Chỉ giữ nhãn `DIABETE4` 1 và 3; mã còn lại bị loại. BMI chia 100; giá trị ngoài miền hợp lệ được đánh dấu thiếu. Mã cột phải được đối chiếu với codebook từng năm.


In [ ]:
def brfss_data(path: str, year: int) -> pd.DataFrame:
    import pyreadstat
    required = set(BRFSS + ['DIABETE4'])
    df, _ = pyreadstat.read_xport(path, usecols=sorted(required))
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{year}: columns missing: {sorted(missing)}; inspect CDC codebook before changing the feature contract')
    df = df[list(BRFSS) + ['DIABETE4']].copy()
    # Exclude gestational diabetes (2), prediabetes (4) and unknown (7, 9, blank) from binary outcome
    df = df[df.DIABETE4.isin([1, 3])].copy()
    df['class'] = df.DIABETE4.map({1: 1, 3: 0})
    for col, valid in CODES.items():
        df[col] = pd.to_numeric(df[col], errors='coerce').where(df[col].isin(valid))
    df['_BMI5'] = pd.to_numeric(df['_BMI5'], errors='coerce') / 100
    df['_BMI5'] = df['_BMI5'].where(df['_BMI5'].between(10, 100))
    return df.drop(columns='DIABETE4')


## 3. Hàm đánh giá

Các hàm chỉ được định nghĩa ở đây; việc tính toán diễn ra sau khi đã huấn luyện.


In [ ]:
def metrics(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    out = {
        'confusion_matrix': confusion_matrix(y, p >= 0.5, labels=[0, 1]).tolist(),
        'classification_report': classification_report(y, p >= 0.5, output_dict=True, zero_division=0)
    }
    out['roc_auc'] = float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None
    out['brier_score'] = float(brier_score_loss(y, p))
    
    # Threshold sweep evaluation
    thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
    sweep = {}
    for th in thresholds:
        pred_th = (p >= th).astype(int)
        rep = classification_report(y, pred_th, output_dict=True, zero_division=0)
        cm = confusion_matrix(y, pred_th, labels=[0, 1]).tolist()
        sweep[str(th)] = {
            'precision_pos': rep['1']['precision'],
            'recall_pos': rep['1']['recall'],
            'f1_pos': rep['1']['f1-score'],
            'accuracy': rep['accuracy'],
            'confusion_matrix': cm
        }
    out['threshold_sweep'] = sweep
    return out


### 3.1. Chỉ số theo nhóm

Nhóm được tính riêng theo loại dữ liệu; kích thước nhóm nhỏ làm chỉ số kém ổn định. Chênh lệch AUC không tự chứng minh tính công bằng.


**UCI:** phân nhóm theo giới và ngưỡng tuổi 50; chỉ tính khi nhóm có đủ mẫu và có cả hai lớp.


In [ ]:
def subgroup_symptoms(test_df, y_true, y_prob):
    subgroups = {}
    for g in ['male', 'female']:
        mask = (test_df['gender'] == g).fillna(False).to_numpy(dtype=bool)
        if mask.sum() >= 5 and len(np.unique(y_true[mask])) == 2:
            subgroups[f'gender_{g}'] = {'count': int(mask.sum()), 'positive_rate': float(np.mean(y_true[mask])), 'roc_auc': float(roc_auc_score(y_true[mask], y_prob[mask])), 'brier_score': float(brier_score_loss(y_true[mask], y_prob[mask]))}
    mask_young = (test_df['age'] < 50).fillna(False).to_numpy(dtype=bool)
    mask_old = (test_df['age'] >= 50).fillna(False).to_numpy(dtype=bool)
    for label, m in [('age_under_50', mask_young), ('age_50_plus', mask_old)]:
        if m.sum() >= 5 and len(np.unique(y_true[m])) == 2:
            subgroups[label] = {'count': int(m.sum()), 'positive_rate': float(np.mean(y_true[m])), 'roc_auc': float(roc_auc_score(y_true[m], y_prob[m])), 'brier_score': float(brier_score_loss(y_true[m], y_prob[m]))}
    return subgroups


**BRFSS:** phân nhóm theo giới, tuổi, hút thuốc và hoạt động thể chất.


In [ ]:
def subgroup_brfss(test_df, y_true, y_prob):
    subgroups = {}
    sex_col = pd.to_numeric(test_df['SEXVAR'], errors='coerce')
    for code, name in [(1, 'male'), (2, 'female')]:
        mask = (sex_col == code).fillna(False).to_numpy(dtype=bool)
        if mask.sum() >= 100:
            subgroups[f'sex_{name}'] = {'count': int(mask.sum()), 'positive_rate': float(np.mean(y_true[mask])), 'roc_auc': float(roc_auc_score(y_true[mask], y_prob[mask])), 'brier_score': float(brier_score_loss(y_true[mask], y_prob[mask]))}
    age_num = pd.to_numeric(test_df['_AGEG5YR'], errors='coerce')
    for label, (low, high) in [('age_18_39', (1, 4)), ('age_40_59', (5, 8)), ('age_60_plus', (9, 13))]:
        mask = age_num.between(low, high).fillna(False).to_numpy(dtype=bool)
        if mask.sum() >= 100:
            subgroups[label] = {'count': int(mask.sum()), 'positive_rate': float(np.mean(y_true[mask])), 'roc_auc': float(roc_auc_score(y_true[mask], y_prob[mask])), 'brier_score': float(brier_score_loss(y_true[mask], y_prob[mask]))}
    smk_num = pd.to_numeric(test_df['_SMOKER3'], errors='coerce')
    for label, cond in [('smoker_current', smk_num.isin([1, 2])), ('smoker_former', smk_num == 3), ('smoker_never', smk_num == 4)]:
        mask = cond.fillna(False).to_numpy(dtype=bool)
        if mask.sum() >= 100:
            subgroups[label] = {'count': int(mask.sum()), 'positive_rate': float(np.mean(y_true[mask])), 'roc_auc': float(roc_auc_score(y_true[mask], y_prob[mask])), 'brier_score': float(brier_score_loss(y_true[mask], y_prob[mask]))}
    act_num = pd.to_numeric(test_df['_TOTINDA'], errors='coerce')
    for label, code in [('physically_active', 1), ('physically_inactive', 2)]:
        mask = (act_num == code).fillna(False).to_numpy(dtype=bool)
        if mask.sum() >= 100:
            subgroups[label] = {'count': int(mask.sum()), 'positive_rate': float(np.mean(y_true[mask])), 'roc_auc': float(roc_auc_score(y_true[mask], y_prob[mask])), 'brier_score': float(brier_score_loss(y_true[mask], y_prob[mask]))}
    return subgroups


Chọn hàm phân nhóm tương ứng với bộ dữ liệu.


In [ ]:
def subgroup_metrics(test_df, y_true, y_prob, kind):
    if kind == 'symptoms':
        return subgroup_symptoms(test_df, y_true, y_prob)
    return subgroup_brfss(test_df, y_true, y_prob)


## 4. Huấn luyện bộ phân loại

Các cell dưới đây là các bước trong `ml/train.py`, được tách để quan sát lần lượt. Khi chưa bật cờ, cell được bỏ qua. Nếu bật cờ, chạy toàn bộ notebook từ đầu để dữ liệu và biến trung gian đồng nhất.


### 4.1. Khởi tạo seed

Cố định seed cho Python, NumPy và PyTorch để tăng khả năng tái lập.


In [ ]:
if RUN_CLASSIFIER:
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    log.info(f'Starting training pipeline for {kind.upper()}...')


### 4.2. Đọc dữ liệu và tạo tập test

UCI loại hồ sơ trùng/xung đột rồi chia test; BRFSS dùng 2021–2022 cho train/validation và 2023 cho test.


In [ ]:
if RUN_CLASSIFIER:
    if kind == 'symptoms':
        df = symptom_data(paths[0])
        source_rows = len(df)
        conflicts = df.groupby(SYMPTOMS, dropna=False)['class'].transform('nunique') > 1
        conflict_rows = int(conflicts.sum())
        df = df.loc[~conflicts].drop_duplicates(subset=SYMPTOMS).copy()
        trainval, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['class'])
        numeric, categorical = (['age'], SYMPTOMS[1:])
        counts = {'source_rows': source_rows, 'conflicting_rows_removed': conflict_rows, 'unique_profiles': len(df)}
    else:
        years = [2021, 2022, 2023]
        frames = [brfss_data(p, y) for p, y in zip(paths, years)]
        trainval = pd.concat(frames[:2], ignore_index=True)
        test = frames[2]
        numeric, categorical = (['_BMI5'], [c for c in BRFSS if c != '_BMI5'])
        for frame in frames:
            for col in categorical:
                frame[col] = frame[col].astype('Int64').astype('string')
        for col in categorical:
            trainval[col] = trainval[col].astype('Int64').astype('string')
        counts = {str(y): len(f) for y, f in zip(years, frames)}


### 4.3. Tách tập train và validation

Chia theo tỷ lệ 80/20 có phân tầng trên phần trainval.


In [ ]:
if RUN_CLASSIFIER:
    train_df, val = train_test_split(trainval, test_size=0.2, random_state=42, stratify=trainval['class'])
    features = numeric + categorical


### 4.4. Xây dựng bộ tiền xử lý

Điền thiếu và chuẩn hóa đặc trưng số; điền thiếu và one-hot cho đặc trưng phân loại.


In [ ]:
if RUN_CLASSIFIER:
    prep = ColumnTransformer([('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric), ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), categorical)], sparse_threshold=0)
    def clean(frame):
        x = frame[features].copy()
        for c in categorical:
            x[c] = x[c].astype(object).where(x[c].notna(), np.nan)
        return x


### 4.5. Fit và biến đổi dữ liệu

Chỉ fit bộ tiền xử lý trên train, sau đó transform validation và test để tránh rò rỉ.


In [ ]:
if RUN_CLASSIFIER:
    xtr = prep.fit_transform(clean(train_df)).astype('float32')
    xv = prep.transform(clean(val)).astype('float32')
    xt = prep.transform(clean(test)).astype('float32')


### 4.6. Chuẩn bị nhãn

Chuyển nhãn train, validation và test thành mảng số nguyên cho các mô hình.


In [ ]:
if RUN_CLASSIFIER:
    ytr = train_df['class'].astype(int).to_numpy()
    yv = val['class'].astype(int).to_numpy()
    yt = test['class'].astype(int).to_numpy()


### 4.7. Huấn luyện baseline Logistic Regression

Baseline có cân bằng trọng số lớp và tạo điểm dự đoán trên test để so sánh.


In [ ]:
if RUN_CLASSIFIER:
    baseline = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42).fit(xtr, ytr)
    p_baseline = baseline.predict_proba(xt)[:, 1]


### 4.8. Khởi tạo MLP, optimizer và hàm mất mát

Adam có learning rate 0,001; BCE dùng trọng số lớp dương. Đầu ra là logit.


In [ ]:
if RUN_CLASSIFIER:
    model = Network(xtr.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    pos_weight = torch.tensor([(len(ytr) - ytr.sum()) / max(ytr.sum(), 1)], dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


### 4.9. Tạo DataLoader và cấu hình dừng sớm

Tập train được xáo trộn theo batch 256; các biến best_loss và patience theo dõi validation.


In [ ]:
if RUN_CLASSIFIER:
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.from_numpy(xtr), torch.tensor(ytr, dtype=torch.float32)), batch_size=256, shuffle=True)
    best_loss = float('inf')
    best = None
    patience = 0


### 4.10. Huấn luyện theo epoch

Mỗi epoch cập nhật từ train, sau đó tính validation loss; dừng khi không cải thiện 7 lượt.


In [ ]:
if RUN_CLASSIFIER:
    for epoch in range(60):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(torch.from_numpy(xv)), torch.tensor(yv, dtype=torch.float32)).item()
        if val_loss < best_loss - 0.0001:
            best_loss = val_loss
            best = {k: v.detach().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= 7:
                break


### 4.11. Tải trọng số tốt nhất và dự đoán test

Dùng sigmoid để lấy điểm MLP; test không tham gia chọn epoch.


In [ ]:
if RUN_CLASSIFIER:
    model.load_state_dict(best)
    model.eval()
    with torch.no_grad():
        p_nn = torch.sigmoid(model(torch.from_numpy(xt))).numpy()


### 4.12. Lưu model và bộ tiền xử lý

Ghi model, baseline và preprocessor vào thư mục kết quả mới, tách khỏi artifacts triển khai.


In [ ]:
if RUN_CLASSIFIER:
    out = Path(output) / kind
    out.mkdir(parents=True, exist_ok=True)
    joblib.dump(prep, out / 'preprocessor.joblib')
    torch.save({'input_size': xtr.shape[1], 'state_dict': model.state_dict()}, out / 'model.pt')
    joblib.dump(baseline, out / 'baseline.joblib')


### 4.13. Đánh giá và lập báo cáo

Tính chỉ số chung, quét ngưỡng và đánh giá theo nhóm; kết quả test chỉ dùng báo cáo, không dùng chọn ngưỡng.


In [ ]:
if RUN_CLASSIFIER:
    subgroup_res = subgroup_metrics(test, yt, p_nn, kind)
    report = {'cohort_counts': counts, 'train': len(train_df), 'validation': len(val), 'test': len(test), 'test_positive_fraction': float(np.mean(yt)), 'features': features, 'neural_network': metrics(yt, p_nn), 'logistic_regression': metrics(yt, p_baseline), 'subgroup_analysis': subgroup_res, 'limitations': 'Educational use only. BRFSS is self-reported, cross-sectional and survey weights are not used in training or these metrics. Not calibrated for medical diagnosis or Vietnamese population. UCI is small (251 unique profiles) and from a single clinic in Sylhet, Bangladesh.'}


### 4.14. Ghi báo cáo JSON

Lưu kích thước tập, đặc trưng, chỉ số và giới hạn của mô hình.


In [ ]:
if RUN_CLASSIFIER:
    (out / 'report.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
    log.info(f"Report saved to {out / 'report.json'}")
    print(json.dumps({'output': str(out), 'counts': counts, 'test': len(test)}, indent=2))


## 5. Học tăng cường DQN

Bài toán mô phỏng bốn nhu cầu học tập; phần thưởng phản ánh quy tắc giả lập, không đo tác động điều trị.


In [ ]:
TOPICS = [
    'Tổng quan về tiểu đường',
    'Ăn uống cân bằng',
    'Vận động thể chất',
    'Theo dõi sức khỏe'
]


### 5.1. Môi trường EducationEnv

Trạng thái gồm bốn cờ, mỗi hành động chọn một chủ đề. Chọn đúng nhu cầu được +1; chọn lặp bị −0,25; episode tối đa bốn bước.


In [ ]:
class EducationEnv(gym.Env):
    """
    Simulated educational environment.
    State: MultiBinary(4) where 1 indicates an unlearned module, 0 indicates completed.
    Action: Discrete(4) corresponding to selecting one module to deliver.
    Reward: +1.0 if the selected module was needed (state[a] == 1).
            -0.25 if the module was already completed (state[a] == 0).
    Episode ends after 4 interaction steps or when all topics are completed.
    """
    def __init__(self):
        super().__init__()
        self.observation_space = spaces.MultiBinary(4)
        self.action_space = spaces.Discrete(4)
        self.state = np.zeros(4, dtype=np.int8)
        self.steps = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        # Generate random initial needs
        self.state = self.np_random.integers(0, 2, size=4, dtype=np.int8)
        self.steps = 0
        return self.state.copy(), {}

    def step(self, action: int):
        # If all topics already learned, action cannot provide new value
        all_done_before = bool(np.all(self.state == 0))
        
        if self.state[action] == 1:
            reward = 1.0
            self.state[action] = 0  # Marked as learned
        else:
            reward = 0.0 if all_done_before else -0.25
            
        self.steps += 1
        terminated = bool(self.steps >= 4 or np.all(self.state == 0))
        truncated = False
        return self.state.copy(), float(reward), terminated, truncated, {}


### 5.2. Hàm đánh giá chính sách

Chạy nhiều episode, lấy trung bình và độ lệch chuẩn tổng thưởng trong một nhóm seed.


In [ ]:
def evaluate_policy(policy_fn, episodes=200, base_seed=10000):
    env = EducationEnv()
    returns = []
    for ep in range(episodes):
        seed = base_seed + ep
        obs, _ = env.reset(seed=seed)
        total_reward = 0.0
        for step in range(4):
            action = policy_fn(obs, seed, step)
            obs, reward, terminated, truncated, _ = env.step(int(action))
            total_reward += reward
            if terminated or truncated:
                break
        returns.append(total_reward)
    return float(np.mean(returns)), float(np.std(returns))


### 5.3. Các chính sách so sánh

Greedy chọn chủ đề đầu tiên còn thiếu, random chọn ngẫu nhiên. Oracle trong mã hiện tại dùng cùng quy tắc với greedy.


In [ ]:
def greedy_heuristic(obs, seed, step):
    unlearned = np.where(obs == 1)[0]
    return unlearned[0] if len(unlearned) > 0 else 0


In [ ]:
def random_policy(obs, seed, step):
    rng = np.random.default_rng(seed * 31 + step)
    return int(rng.integers(4))


In [ ]:
def optimal_oracle(obs, seed, step):
    unlearned = np.where(obs == 1)[0]
    if len(unlearned) > 0:
        return int(unlearned[0])
    return 0


### 5.4. Khởi tạo môi trường và DQN

DQN dùng seed 42, replay buffer 10.000, batch 64 và hệ số chiết khấu 0,99. Bật `RUN_RL` trước khi chạy lại từ đầu.


In [ ]:
if RUN_RL:
    output_dir = str(OUT / 'rl')
    total_timesteps = 30000
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    log.info('Initializing DQN training on EducationEnv...')
    train_env = EducationEnv()
    model = DQN('MlpPolicy', train_env, seed=42, learning_starts=200, buffer_size=10000, batch_size=64, gamma=0.99, verbose=0)


### 5.5. Huấn luyện và lưu chính sách

DQN tương tác với môi trường trong 30.000 timestep. Tệp được lưu trong thư mục riêng của notebook.


In [ ]:
if RUN_RL:
    model.learn(total_timesteps=total_timesteps)
    model.save(str(out / 'education_policy'))
    log.info(f"DQN policy saved to {out / 'education_policy.zip'}")


### 5.6. Chuẩn bị các nhóm seed đánh giá

Năm nhóm seed được dùng để so sánh cùng chính sách trên 200 episode mỗi nhóm.


In [ ]:
if RUN_RL:
    eval_seeds = [1000, 2000, 3000, 4000, 5000]
    dqn_means, dqn_stds = ([], [])
    random_means, random_stds = ([], [])
    greedy_means, greedy_stds = ([], [])
    optimal_means, optimal_stds = ([], [])
    dqn_policy = lambda obs, s, step: model.predict(obs, deterministic=True)[0]


### 5.7. So sánh DQN và baseline

Chạy lần lượt DQN, greedy, random và oracle; lưu điểm trung bình cho từng nhóm seed.


In [ ]:
if RUN_RL:
    for s in eval_seeds:
        m_dqn, s_dqn = evaluate_policy(dqn_policy, episodes=200, base_seed=s)
        m_rnd, s_rnd = evaluate_policy(random_policy, episodes=200, base_seed=s)
        m_grd, s_grd = evaluate_policy(greedy_heuristic, episodes=200, base_seed=s)
        m_opt, s_opt = evaluate_policy(optimal_oracle, episodes=200, base_seed=s)
        dqn_means.append(m_dqn)
        random_means.append(m_rnd)
        greedy_means.append(m_grd)
        optimal_means.append(m_opt)


### 5.8. Lưu báo cáo RL

Tổng hợp trung bình và độ lệch chuẩn giữa các nhóm seed, sau đó ghi `report.json`.


In [ ]:
if RUN_RL:
    report = {'episodes_per_seed': 200, 'evaluation_seeds': eval_seeds, 'topics': TOPICS, 'state_definition': 'Vector nhị phân 4 chiều [Tổng quan, Ăn uống, Vận động, Theo dõi]; 1 = cần học, 0 = đã hoàn tất.', 'action_definition': 'Chọn 1 trong 4 chủ đề giáo dục tương ứng với chỉ số [0, 1, 2, 3].', 'reward_definition': '+1.0 nếu chủ đề đang cần học; -0.25 nếu chọn trùng chủ đề đã hoàn tất; 0.0 khi hoàn tất hết.', 'results': {'dqn_policy': {'mean_return': float(np.mean(dqn_means)), 'std_return': float(np.std(dqn_means))}, 'greedy_heuristic': {'mean_return': float(np.mean(greedy_means)), 'std_return': float(np.std(greedy_means))}, 'random_baseline': {'mean_return': float(np.mean(random_means)), 'std_return': float(np.std(random_means))}, 'optimal_oracle': {'mean_return': float(np.mean(optimal_means)), 'std_return': float(np.std(optimal_means))}}, 'warning': 'Synthetic reward; simulated educational task, zero evidence of clinical benefit or medical treatment.'}
    (out / 'report.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
    print(json.dumps(report, indent=2, ensure_ascii=False))


## 6. Xem kết quả đã lưu trước đó

Các cell sau đọc `artifacts/` của dự án; các số này không phải kết quả huấn luyện lại trong phiên notebook hiện tại.


In [ ]:
for saved_kind in ['symptoms', 'brfss']:
    saved_report = json.loads((ROOT / 'artifacts' / saved_kind / 'report.json').read_text())
    print(f"\n{saved_kind}: train={saved_report['train']}, validation={saved_report['validation']}, test={saved_report['test']}")
    for model_name in ['neural_network', 'logistic_regression']:
        result = saved_report[model_name]
        print(model_name, 'accuracy=', round(result['classification_report']['accuracy'], 4),
              'AUC=', round(result['roc_auc'], 4), 'CM=', result['confusion_matrix'])


### 6.1. Kết quả RL đã lưu

So sánh reward giữa các chính sách. DQN bằng heuristic trong môi trường hiện tại, không có bằng chứng hiệu quả y khoa.


In [ ]:
rl_report = json.loads((ROOT / 'artifacts/rl/report.json').read_text())
for policy, result in rl_report['results'].items():
    print(policy, result)


## 7. Diễn giải và giới hạn

- UCI chỉ 251 hồ sơ duy nhất từ một cơ sở; tập test nhỏ. BRFSS là dữ liệu cắt ngang tự báo cáo, không đại diện trực tiếp dân số Việt Nam.
- Các kết quả test không chứng minh hiệu quả lâm sàng; không dùng điểm làm chẩn đoán hay điều trị.
- Với DQN, heuristic đơn giản đạt cùng điểm trung bình trong môi trường này; không khẳng định DQN tốt hơn.
- Pipeline triển khai và notebook chia sẻ cùng mã gốc tại thời điểm biên soạn; khi sửa mã gốc cần cập nhật notebook.
